# 01 - Universe Screening

**Author:** Sacha Huberty

**Purpose:** Build the ATLAS L0 investable universe from a broad, global candidate pool of ETFs. Applies history/liquidity/missing-data screens (S6), computes trailing statistical ratios and factor correlations, de-duplicates highly correlated assets within each asset-class bucket via K-Means clustering, and persists a versioned universe snapshot for downstream stages to consume.

**Last updated:** 2026-07-24

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:.4f}'.format

from atlas import data, universe

cfg = data.load_config()
cfg["general"]

## Data

In [ ]:
# Screening is re-run per walk-forward fold using as-of data only.
# For this exploratory pass we snapshot at the in-sample boundary so
# nothing out-of-sample leaks into the design phase.
candidates = universe.BUILTIN_CANDIDATE_POOL
tickers = sorted(candidates.keys())
as_of = pd.Timestamp(cfg["general"]["is_end_date"])

prices = data.download_prices(
    tickers,
    start=cfg["general"]["start_date"],
    end=as_of + pd.Timedelta(days=1),
)
prices = data.align_calendars(prices)
prices.tail()

In [ ]:
normalized = prices / prices.iloc[0]
normalized.plot(figsize=(11, 5), title="Candidate pool: normalized prices")
plt.ylabel("Growth of $1")
plt.show()

## Analysis / signal logic

In [ ]:
# S6 statistical ratios: 1/3/5Y trailing return, vol, Sharpe per asset
ratios = universe.stat_ratios(prices)
ratios.sort_values("sharpe_3y", ascending=False)

In [ ]:
# Correlation vs equity/fixed-income/cash factor proxies (S6)
returns = data.daily_returns(prices)
factor_tickers = list(universe.FACTOR_PROXIES.values())
factors = returns[factor_tickers]
asset_returns = returns.drop(columns=factor_tickers)
factor_corr = universe.factor_correlations(asset_returns, factors)
factor_corr

In [ ]:
# Full screen: history/missing-data filters + K-Means correlation
# de-dup per asset-class bucket (S6)
screened_universe = universe.screen(candidates, prices, cfg, as_of)
screened_universe

In [ ]:
saved_path = universe.save_universe(screened_universe, as_of)
saved_path

## Results

In [ ]:
screened_universe.groupby("class_bucket").size().rename("n_assets")

In [ ]:
print(
    f"Kept {len(screened_universe)} of {len(candidates)} candidates "
    f"as of {as_of.date()}."
)

## Notes / next steps

- Liquidity is currently approximated by trading-day coverage (missing-data ratio); a true average-dollar-volume screen needs `Volume`, not just `Close`, cached in `data.py` -- add when the universe grows past the built-in pool.
- With the default `n_asset_clusters: 8` and at most 8-9 candidates per bucket, K-Means de-dup is close to a no-op here (every asset can be its own cluster). It starts pruning once a bucket's candidate count exceeds the cluster count -- worth revisiting once the candidate pool is widened beyond the built-in ETFs.
- This snapshot uses `is_end_date` as `as_of` to keep the design phase strictly in-sample; the OOS walk-forward will re-screen per fold using its own `as_of` date, never peeking ahead.
- Next (stage 2): `allocation.py` classical methods (max Sharpe, GMV, Risk Parity, HRP) and `backtest.py`'s weekly engine, run OOS against this saved universe as the baseline.